
# Multi-hop Incoming Citation Maps — **Who Cited the Citers** (Depth 1–3)

This notebook builds **incoming** citation networks up to a chosen depth for two DOI sets (Set A / Set B).  
Controls: `IN_DEPTH`, `MAX_CITERS_PER_NODE`, `NODES_HARD_CAP`.  
Labels remain **source-only** for clarity.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [1]:

import requests, time, json, os, re, collections
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

LABEL_A = "SetA"
LABEL_B = "SetB"

DOIS_SET_A = [
    "10.15640/ijn.v7n1a2",
    "10.2174/1874434602014010168",
    "10.3390/nursrep10020023",
    "10.4236/ojn.2020.107050",
]

DOIS_SET_B = [
    "10.1016/j.ijnurstu.2020.103842",
    "10.1097/NCC.0000000000000693",
]

IN_DEPTH = 2
MAX_CITERS_PER_NODE = 30
NODES_HARD_CAP = 5000

SLEEP = 0.5
OUT_BASE_DIR = "citation_maps_output_multihop_IN"
os.makedirs(OUT_BASE_DIR, exist_ok=True)

def oa_get(url):
    r = requests.get(url, timeout=45)
    r.raise_for_status()
    return r.json()

def work_from_doi(doi):
    url = f"https://api.openalex.org/works/https://doi.org/{doi.strip().lower()}"
    return oa_get(url)

def get_citers(openalex_id, per_page=25):
    base = f"https://api.openalex.org/works?filter=cites:{openalex_id}&per_page={per_page}"
    data = oa_get(base)
    return data.get("results", [])

def safe_list(x):
    return x if isinstance(x, list) else []

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def doi_str(x):
    if not x:
        return ""
    if isinstance(x, str) and x.lower().startswith("https://doi.org/"):
        return x.split("https://doi.org/")[-1].strip()
    return x.strip()

def slugify(text):
    return re.sub(r'[^A-Za-z0-9._-]+', '_', text)

def draw_sources_only(GX, title, path):
    plt.figure(figsize=(12, 10))
    pos = nx.spring_layout(GX, seed=42, k=0.7 if GX.number_of_nodes() < 800 else None)

    focal_nodes = [n for n, d in GX.nodes(data=True) if d.get("type") == "focal"]
    other_nodes = [n for n in GX.nodes() if n not in focal_nodes]

    nx.draw_networkx_nodes(GX, pos, nodelist=other_nodes, node_size=28)
    nx.draw_networkx_edges(GX, pos, arrows=True, arrowstyle="-|>", arrowsize=10, width=0.35)

    nx.draw_networkx_nodes(GX, pos, nodelist=focal_nodes, node_size=650, node_shape="s")
    labels = {n: (GX.nodes[n].get("doi") or n.split("/")[-1]) for n in focal_nodes}
    nx.draw_networkx_labels(GX, pos, labels=labels, font_size=8)

    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(path, dpi=170)
    plt.close()


In [2]:

def expand_incoming_multihop(set_label, doi_list):
    out_dir = os.path.join(OUT_BASE_DIR, set_label)
    ego_dir = os.path.join(out_dir, f"ego_IN_depth{IN_DEPTH}")
    ensure_dir(out_dir)
    ensure_dir(ego_dir)

    G = nx.DiGraph()
    depth_by_node = {}
    focal_ids = []

    for doi in doi_list:
        try:
            w = work_from_doi(doi)
        except Exception as e:
            print(f"[{set_label}] Failed DOI {doi}: {e}")
            continue
        fid = w.get("id", "")
        focal_ids.append(fid)
        G.add_node(fid, type="focal", doi=doi_str(w.get("doi")) or doi, set=set_label)
        depth_by_node[fid] = 0
        time.sleep(SLEEP)

    frontier = list(focal_ids)
    visited_for_citers = set()
    current_depth = 0

    while current_depth < IN_DEPTH and len(G) < NODES_HARD_CAP:
        next_frontier = []
        for target in frontier:
            if target in visited_for_citers:
                continue
            visited_for_citers.add(target)

            if len(G) >= NODES_HARD_CAP:
                break

            try:
                citers = get_citers(target, per_page=MAX_CITERS_PER_NODE)
            except Exception as e:
                print(f"[{set_label}] Citers fetch failed for {target}: {e}")
                continue

            for cw in citers:
                cid = cw.get("id", "")
                if not cid:
                    continue

                if cid not in G:
                    G.add_node(cid, type=f"in_level{current_depth+1}", doi=doi_str(cw.get('doi')) or "", set=set_label)
                    depth_by_node[cid] = current_depth + 1

                if not G.has_edge(cid, target):
                    G.add_edge(cid, target, relation="cites")

                if current_depth + 1 < IN_DEPTH:
                    next_frontier.append(cid)

            time.sleep(SLEEP)

        frontier = list(set(next_frontier))
        current_depth += 1

        if len(G) >= NODES_HARD_CAP:
            print(f"[{set_label}] Reached node hard cap ({NODES_HARD_CAP}); stopping expansion.")
            break

    combined_png = os.path.join(out_dir, f"{set_label}_combined_IN_depth{IN_DEPTH}.png")
    draw_sources_only(G, f"{set_label} Incoming citations up to depth {IN_DEPTH} (labels = source DOIs only)", combined_png)

    # Per-source ego IN maps
    for fid in focal_ids:
        keep = {fid}
        for n, d in depth_by_node.items():
            if d >= 1 and d <= IN_DEPTH:
                keep.add(n)
        G_ego = G.subgraph(keep).copy()

        ego_png = os.path.join(ego_dir, f"ego_IN_depth{IN_DEPTH}_{slugify(G.nodes[fid].get('doi') or fid.split('/')[-1])}.png")
        draw_sources_only(G_ego, f"IN ego (depth {IN_DEPTH}) to {G.nodes[fid].get('doi')}", ego_png)

    # CSVs
    depth_counts = collections.Counter(depth_by_node.values())
    depth_rows = [{"depth": d, "count": c} for d, c in sorted(depth_counts.items())]
    pd.DataFrame(depth_rows).to_csv(os.path.join(out_dir, f"{set_label}_nodes_by_depth.csv"), index=False)

    indeg = dict(G.in_degree())
    rows = sorted([{"node": n, "in_degree": indeg[n], "type": G.nodes[n].get("type",""), "doi": G.nodes[n].get("doi","")} for n in G.nodes()], key=lambda x: x["in_degree"], reverse=True)
    pd.DataFrame(rows[:50]).to_csv(os.path.join(out_dir, f"{set_label}_top_hubs.csv"), index=False)

    pd.DataFrame([{
        "set": set_label,
        "depth": IN_DEPTH,
        "focal_count": len(focal_ids),
        "total_nodes": G.number_of_nodes(),
        "total_edges": G.number_of_edges(),
        "hard_cap_reached": int(G.number_of_nodes() >= NODES_HARD_CAP)
    }]).to_csv(os.path.join(out_dir, f"{set_label}_summary.csv"), index=False)

    print(f"[{set_label}] Saved combined IN-depth map and per-source IN ego maps.")
    return G


In [3]:

G_A = expand_incoming_multihop(LABEL_A, DOIS_SET_A)
G_B = expand_incoming_multihop(LABEL_B, DOIS_SET_B)
print("Done. Root output folder:", OUT_BASE_DIR)


[SetA] Saved combined IN-depth map and per-source IN ego maps.
[SetB] Saved combined IN-depth map and per-source IN ego maps.
Done. Root output folder: citation_maps_output_multihop_IN
